In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

# 1. CONFIG
@dataclass
class Config:
    vocab_size: int = 50257
    block_size: int = 2048
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: int = 4
    n_embd: int = 768
    rope_theta: float = 500000.0
    norm_eps: float = 1e-6

# 2. RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * self.scale / (x.pow(2).mean(-1, keepdim=True) + self.eps).sqrt()

# 3. RoPE precompute
def precompute_rope_freqs(dim, max_len, theta=500000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[:dim//2].float() / dim))
    t = torch.arange(max_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)

# 4. RoPE apply
def apply_rotary_emb(q, k, cos, sin, position_ids):
    cos = cos[position_ids].unsqueeze(1)
    sin = sin[position_ids].unsqueeze(1)
    
    head_dim = q.shape[-1]
    q_real, q_imag = q[..., :head_dim//2], q[..., head_dim//2:]
    k_real, k_imag = k[..., :head_dim//2], k[..., head_dim//2:]
    
    q_rot = torch.cat((q_real * cos - q_imag * sin, q_real * sin + q_imag * cos), dim=-1)
    k_rot = torch.cat((k_real * cos - k_imag * sin, k_real * sin + k_imag * cos), dim=-1)
    return q_rot, k_rot

# 5. BLOCK (Chunked GQA Implementation to prevent OOM)
class AmadeusZeroBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        
        assert self.n_head % self.n_kv_head == 0
        self.num_key_value_groups = self.n_head // self.n_kv_head
        
        hidden_dim = int(8 * config.n_embd / 3)
        hidden_dim = ((hidden_dim + 255) // 256) * 256
        
        self.ln_1 = RMSNorm(config.n_embd, eps=config.norm_eps)
        self.ln_2 = RMSNorm(config.n_embd, eps=config.norm_eps)
        
        self.q_size = self.n_head * self.head_dim
        self.kv_size = self.n_kv_head * self.head_dim
        self.c_attn = nn.Linear(config.n_embd, self.q_size + 2 * self.kv_size, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        
        self.mlp = nn.ModuleDict({
            'gate_proj': nn.Linear(config.n_embd, hidden_dim, bias=False),
            'up_proj': nn.Linear(config.n_embd, hidden_dim, bias=False),
            'down_proj': nn.Linear(hidden_dim, config.n_embd, bias=False),
        })

    def forward(self, x, cos, sin, position_ids):
        x = x + self._attn_block(self.ln_1(x), cos, sin, position_ids)
        x = x + self._mlp_block(self.ln_2(x))
        return x

    def _attn_block(self, x, cos, sin, position_ids):
        B, T, C = x.size()
        
        qkv = self.c_attn(x)
        q, k, v = qkv.split([self.q_size, self.kv_size, self.kv_size], dim=2)
        
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        
        q, k = apply_rotary_emb(q, k, cos, sin, position_ids)

        # --------------------------------------------------------
        # THE FIX: Chunked GQA Math
        # We loop through the 4 KV heads sequentially. 
        # By using .expand() inside the loop, we map the memory 
        # without ever duplicating it physically.
        # --------------------------------------------------------
        y_chunks = []
        for i in range(self.n_kv_head):
            # Grab 3 Query heads
            q_chunk = q[:, i * self.num_key_value_groups : (i + 1) * self.num_key_value_groups, :, :]
            
            # Grab 1 KV head and expand it to match the 3 Query heads (zero memory copy)
            k_chunk = k[:, i : i + 1, :, :].expand(-1, self.num_key_value_groups, -1, -1)
            v_chunk = v[:, i : i + 1, :, :].expand(-1, self.num_key_value_groups, -1, -1)
            
            # Run SDPA on this small fraction of the data
            y_chunk = F.scaled_dot_product_attention(q_chunk, k_chunk, v_chunk, is_causal=True)
            y_chunks.append(y_chunk)

        # Recombine the 4 chunks back into a single 12-head tensor
        y = torch.cat(y_chunks, dim=1)
        # --------------------------------------------------------

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

    def _mlp_block(self, x):
        gate = F.silu(self.mlp.gate_proj(x))
        up = self.mlp.up_proj(x)
        return self.mlp.down_proj(gate * up)

# 6. MAIN MODEL
class AmadeusZeroLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(config.vocab_size, config.n_embd),
            'h': nn.ModuleList([AmadeusZeroBlock(config) for _ in range(config.n_layer)]),
            'ln_f': RMSNorm(config.n_embd, eps=config.norm_eps),
        })
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight
        
        dim = config.n_embd // config.n_head
        max_len = config.block_size * 2
        cos, sin = precompute_rope_freqs(dim, max_len, theta=config.rope_theta)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        std = 0.02
        if isinstance(module, nn.Linear):
            if module.weight.size(0) == self.config.n_embd:
                std = 0.02 / math.sqrt(2 * self.config.n_layer)
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, position_ids=None):
        _, t = idx.size()
        assert t <= self.config.block_size
        
        if position_ids is None:
            position_ids = torch.arange(t, dtype=torch.long, device=idx.device).unsqueeze(0)
            
        tok_emb = self.transformer.wte(idx)
        
        x = tok_emb
        for block in self.transformer.h:
            x = block(x, self.cos, self.sin, position_ids)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [2]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(f"Vocabulary size: {tokenizer.n_vocab:,}")
print(f"Special tokens: {tokenizer.special_tokens_set}")

Vocabulary size: 50,257
Special tokens: {'<|endoftext|>'}


In [3]:
print(tokenizer.encode("<|endoftext|>", allowed_special="all"))

[50256]


```python
# OLD CODE - Commented out for new train.py approach
from accelerate import Accelerator

conf = Config()
model = AmadeusZeroLM(conf)

# Initialize Accelerate (This permanently replaces DataParallel)
accelerator = Accelerator(mixed_precision="fp16")
device = accelerator.device

# Move the raw model to the device managed by Accelerate
model = model.to(device)

accelerator.print(f"Active Device: {device}")
```

```


```python
# OLD CODE - Commented out for new train.py approach
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name:20} | Shape: {str(param.shape):20} | Params: {param.numel():,}")
```


```python
# OLD CODE - Commented out for new train.py approach
def count_parameters(model):
    # Sum semua elemen di tiap parameter yang butuh gradien
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'AmadeusZeroLM Parameter: {count_parameters(model):,}')
```


In [7]:
conf = Config()
assert isinstance(conf.vocab_size, int), "CONFIG Broken!"
print("Config OK:", conf.vocab_size)  

Config OK: 50257


In [8]:
import os
from kaggle_secrets import UserSecretsClient

# Fetch the secret key from Kaggle Secrets
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGINGFACE_KEY")

# Set the environment variable for Hugging Face integration
os.environ["HF_TOKEN"] = secret_value_0

In [9]:
from huggingface_hub import hf_hub_download
import pandas as pd
# 2. Download the correct parquet file from the repository
repo_id = "karpathy/tinystories-gpt4-clean"
filename = "tinystories_gpt4_clean.parquet"

print(f"Downloading {filename} from {repo_id}...")
downloaded_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="dataset"
)

# 3. Process the Parquet file using pandas
# Reading the binary format correctly into a structured DataFrame
df = pd.read_parquet(downloaded_path)

# Extracting the text column (assuming 'text' or 'story' is the column name)
# If the column name is different, we grab the first column automatically
text_column = df.columns[0]
stories = df[text_column].tolist()

print("First story (300 chars):\n")
story = stories[0]
print(story.strip()[:300], "...")

print(f"\nTotal number of stories: {len(stories):,}")

tinystories_gpt4_clean.parquet:   0%|          | 0.00/673M [00:00<?, ?B/s]

First story (300 chars):

Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.
Jack wanted to get on the wheel, so he ran to it. He put his hands on it and ge ...

Total number of stories: 2,732,634


```python
# OLD CODE - Commented out for new train.py approach
import os
import math
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Custom Dataset for tokenizing the stories (Unchanged)
class StoryDataset(Dataset):
    def __init__(self, texts, tokenizer, block_size):
        self.texts = texts
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.pad_token_id = 50256 

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = self.tokenizer.encode(text, allowed_special="all")
        
        max_len = self.block_size + 1
        
        if len(tokens) < max_len:
            tokens.extend([self.pad_token_id] * (max_len - len(tokens)))
        else:
            tokens = tokens[:max_len]
            
        x = torch.tensor(tokens[:-1], dtype=torch.long)
        y = torch.tensor(tokens[1:], dtype=torch.long)
        
        return x, y

# 2. Dataloaders (Increased batch_size to utilize the freed-up VRAM)
train_dataset = StoryDataset(stories[:10000], tokenizer, conf.block_size)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
total_steps = len(train_loader)

val_dataset = StoryDataset(stories[2600000:], tokenizer, conf.block_size)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=2)

# 3. Setup Optimizer & Scheduler
epochs = 1
learning_rate = 3e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

# 4. The Accelerate Wrapper (Crucial for DDP)
model, optimizer, train_loader, val_loader, scheduler = accelerator.prepare(
    model, optimizer, train_loader, val_loader, scheduler
)

# 5. Safe Multi-GPU Printing
accelerator.print(f"Total stories in dataset: {len(train_dataset):,}")
accelerator.print(f"Batch size per GPU: {train_loader.batch_size}")
accelerator.print(f"Total steps per epoch actually: {total_steps:,}")
accelerator.print(f"Total stories in validation set: {len(val_dataset):,}")
```

```


```python
# OLD CODE - Commented out for new train.py approach
# 3. Setup the Optimizer & Scheduler
epochs = 1
learning_rate = 3e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

# 4. Pass EVERYTHING through Accelerate to distribute the load evenly across T4s
model, optimizer, train_loader, val_loader, scheduler = accelerator.prepare(
    model, optimizer, train_loader, val_loader, scheduler
)

# 5. Checkpoint Settings
start_epoch = 0
start_step = 0

accelerator.print("Checking for checkpoint on Hugging Face...")
try:
    checkpoint_path = hf_hub_download(
        repo_id="RinKana/AmadeusZero-114M", 
        filename="kaggle/working/amadeus_final.pt"
    )
    accelerator.print(f"Found checkpoint at {checkpoint_path}. Loading...")
    
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    # Unwrap model before loading state dict to avoid DDP prefix mismatches
    accelerator.unwrap_model(model).load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    for param_group in optimizer.param_groups:
        param_group['lr'] = learning_rate
        param_group['initial_lr'] = learning_rate
        
    start_epoch = checkpoint['epoch']
    saved_step = checkpoint['step']
    
    STARTING_NEW_EPOCH = True 
    
    if STARTING_NEW_EPOCH:
        accelerator.print("Initializing new epoch with dynamic dataset size. Resetting steps.")
        start_epoch += 1
        start_step = 0
    else:
        start_step = saved_step + 1 
        
    accelerator.print(f"Successfully resumed. Targeting Epoch {start_epoch}, starting at Step {start_step}")
    
except Exception as e:
    accelerator.print(f"No checkpoint found or download failed. Starting training from scratch.")

# 6. Estimation Function
@torch.no_grad()
def estimate_loss(model, dataloader, eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters, device=device)
    for k, (x, y) in enumerate(dataloader):
        if k >= eval_iters:
            break
        
        # Accelerate handles device mapping automatically for prepared dataloaders
        with accelerator.autocast():
            logits, loss = model(x, targets=y)
            
        losses[k] = loss.mean().item()
        del logits, loss
        
    model.train()
    return losses.mean().item()

# 7. Training Loop
model.train()

accelerator.print("Starting training loop...")
for epoch in range(start_epoch, epochs):
    for step, (x, y) in enumerate(train_loader):
        if epoch == start_epoch and step < start_step:
            continue
            
        # Forward pass using accelerator's mixed precision context
        with accelerator.autocast():
            logits, loss = model(x, targets=y)
        
        loss = loss.mean()
        perplexity = torch.exp(loss)
        
        train_loss_val = loss.item()
        train_ppl_val = perplexity.item()
        
        optimizer.zero_grad(set_to_none=True)
        
        # THE FIX: Use accelerate's backward pass so it scales the fp16 gradients!
        accelerator.backward(loss)
        
        # Use accelerate's gradient clipping for stability
        accelerator.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()
        
        del logits, loss, perplexity
        
        if step % 1000 == 0:
            torch.cuda.empty_cache()
            
            val_loss = estimate_loss(model, val_loader, eval_iters=50)
            val_perplexity = math.exp(val_loss)
            current_lr = scheduler.get_last_lr()[0]
            
            # Only print on the main GPU process to avoid double logging
            accelerator.print(f"Epoch: {epoch} | Step: {step} | LR: {current_lr:.6f} | Train Loss: {train_loss_val:.4f} | Val Loss: {val_loss:.4f} | Val PPL: {val_perplexity:.4f} | Perplexity: {train_ppl_val:.4f}") 
            
        is_last_step = step == len(train_loader) - 1
        
        # Only the main process (GPU 0) should handle the file writing
        if accelerator.is_main_process:
            if (step > 0 and step % 10000 == 0) or is_last_step:
                accelerator.print(f"Saving checkpoint at Epoch {epoch}, Step {step}...")
                
                if step % 1000 != 0:
                    torch.cuda.empty_cache()
                    val_loss = estimate_loss(model, val_loader, eval_iters=50)
                    val_perplexity = math.exp(val_loss)

                # THE FIX: Unwrap the model to save raw weights, not DDP wrappers
                unwrapped_model = accelerator.unwrap_model(model)
                
                torch.save({
                    'epoch': epoch,
                    'step': step,
                    'model_state_dict': unwrapped_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'loss': train_loss_val,
                    'perplexity': train_ppl_val,
                    'val_loss': val_loss,
                    'val_perplexity': val_perplexity,
                }, "amadeus_final.pt")

if accelerator.is_main_process:
    accelerator.print("Training complete. Final checkpoint saved.")
```

```


In [ ]:
%%writefile train.py
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from huggingface_hub import hf_hub_download
import tiktoken
from accelerate import Accelerator
from tqdm.auto import tqdm

# 1. CONFIG
@dataclass
class Config:
    vocab_size: int = 50257
    block_size: int = 2048
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: int = 4
    n_embd: int = 768
    rope_theta: float = 500000.0
    norm_eps: float = 1e-6

# 2. RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * self.scale / (x.pow(2).mean(-1, keepdim=True) + self.eps).sqrt()

# 3. RoPE precompute
def precompute_rope_freqs(dim, max_len, theta=500000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[:dim//2].float() / dim))
    t = torch.arange(max_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)

# 4. RoPE apply
def apply_rotary_emb(q, k, cos, sin, position_ids):
    cos = cos[position_ids].unsqueeze(1)
    sin = sin[position_ids].unsqueeze(1)
    
    head_dim = q.shape[-1]
    q_real, q_imag = q[..., :head_dim//2], q[..., head_dim//2:]
    k_real, k_imag = k[..., :head_dim//2], k[..., head_dim//2:]
    
    q_rot = torch.cat((q_real * cos - q_imag * sin, q_real * sin + q_imag * cos), dim=-1)
    k_rot = torch.cat((k_real * cos - k_imag * sin, k_real * sin + k_imag * cos), dim=-1)
    return q_rot, k_rot

# 5. BLOCK
class AmadeusZeroBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        
        assert self.n_head % self.n_kv_head == 0
        self.num_key_value_groups = self.n_head // self.n_kv_head
        
        hidden_dim = int(8 * config.n_embd / 3)
        hidden_dim = ((hidden_dim + 255) // 256) * 256
        
        self.ln_1 = RMSNorm(config.n_embd, eps=config.norm_eps)
        self.ln_2 = RMSNorm(config.n_embd, eps=config.norm_eps)
        
        self.q_size = self.n_head * self.head_dim
        self.kv_size = self.n_kv_head * self.head_dim
        self.c_attn = nn.Linear(config.n_embd, self.q_size + 2 * self.kv_size, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        
        self.mlp = nn.ModuleDict({
            'gate_proj': nn.Linear(config.n_embd, hidden_dim, bias=False),
            'up_proj': nn.Linear(config.n_embd, hidden_dim, bias=False),
            'down_proj': nn.Linear(hidden_dim, config.n_embd, bias=False),
        })

    def forward(self, x, cos, sin, position_ids):
        x = x + self._attn_block(self.ln_1(x), cos, sin, position_ids)
        x = x + self._mlp_block(self.ln_2(x))
        return x

    def _attn_block(self, x, cos, sin, position_ids):
        B, T, C = x.size()
        
        qkv = self.c_attn(x)
        q, k, v = qkv.split([self.q_size, self.kv_size, self.kv_size], dim=2)
        
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        
        q, k = apply_rotary_emb(q, k, cos, sin, position_ids)

        y_chunks = []
        for i in range(self.n_kv_head):
            q_chunk = q[:, i * self.num_key_value_groups : (i + 1) * self.num_key_value_groups, :, :]
            k_chunk = k[:, i : i + 1, :, :].expand(-1, self.num_key_value_groups, -1, -1)
            v_chunk = v[:, i : i + 1, :, :].expand(-1, self.num_key_value_groups, -1, -1)
            y_chunk = F.scaled_dot_product_attention(q_chunk, k_chunk, v_chunk, is_causal=True)
            y_chunks.append(y_chunk)

        y = torch.cat(y_chunks, dim=1)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

    def _mlp_block(self, x):
        gate = F.silu(self.mlp.gate_proj(x))
        up = self.mlp.up_proj(x)
        return self.mlp.down_proj(gate * up)

# 6. MAIN MODEL
class AmadeusZeroLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(config.vocab_size, config.n_embd),
            'h': nn.ModuleList([AmadeusZeroBlock(config) for _ in range(config.n_layer)]),
            'ln_f': RMSNorm(config.n_embd, eps=config.norm_eps),
        })
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight
        
        dim = config.n_embd // config.n_head
        max_len = config.block_size * 2
        cos, sin = precompute_rope_freqs(dim, max_len, theta=config.rope_theta)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        std = 0.02
        if isinstance(module, nn.Linear):
            if module.weight.size(0) == self.config.n_embd:
                std = 0.02 / math.sqrt(2 * self.config.n_layer)
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, position_ids=None):
        _, t = idx.size()
        assert t <= self.config.block_size
        
        if position_ids is None:
            position_ids = torch.arange(t, dtype=torch.long, device=idx.device).unsqueeze(0)
            
        tok_emb = self.transformer.wte(idx)
        
        x = tok_emb
        for block in self.transformer.h:
            x = block(x, self.cos, self.sin, position_ids)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# 7. Dataset
class StoryDataset(Dataset):
    def __init__(self, texts, tokenizer, block_size):
        self.texts = texts
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.pad_token_id = 50256 

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = self.tokenizer.encode(text, allowed_special="all")
        
        max_len = self.block_size + 1
        
        if len(tokens) < max_len:
            tokens.extend([self.pad_token_id] * (max_len - len(tokens)))
        else:
            tokens = tokens[:max_len]
            
        x = torch.tensor(tokens[:-1], dtype=torch.long)
        y = torch.tensor(tokens[1:], dtype=torch.long)
        
        return x, y

def main():
    # Initialize Accelerate for Multi-GPU training
    accelerator = Accelerator(mixed_precision="fp16")
    device = accelerator.device
    
    accelerator.print("Starting environment setup...")
    
    # Download dataset on the main process only to prevent race conditions
    with accelerator.main_process_first():
        repo_id = "karpathy/tinystories-gpt4-clean"
        filename = "tinystories_gpt4_clean.parquet"
        downloaded_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")
        df = pd.read_parquet(downloaded_path)
        text_column = df.columns[0]
        stories = df[text_column].tolist()
        
    tokenizer = tiktoken.get_encoding("gpt2")
    conf = Config()
    
    # Create Dataloaders
    # We slice the dataset identically on all processes, Accelerate will automatically shard it.
    train_dataset = StoryDataset(stories[:10000], tokenizer, conf.block_size)
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
    
    val_dataset = StoryDataset(stories[2600000:], tokenizer, conf.block_size)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=2)
    
    model = AmadeusZeroLM(conf)
    
    epochs = 1
    learning_rate = 3e-4
    total_steps = len(train_loader)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
    
    # Pass EVERYTHING through Accelerate to distribute the load across GPUs
    model, optimizer, train_loader, val_loader, scheduler = accelerator.prepare(
        model, optimizer, train_loader, val_loader, scheduler
    )
    
    accelerator.print(f"Total stories in dataset: {len(train_dataset):,}")
    accelerator.print(f"Total steps per epoch (after sharding): {len(train_loader):,}")
    
    # Continuous Epoch Logic / Checkpoint Resuming
    start_epoch = 0
    start_step = 0
    checkpoint_file = "amadeus_final.pt"
    
    # 1. Try local checkpoint first
    if os.path.exists(checkpoint_file):
        accelerator.print(f"Found local checkpoint at {checkpoint_file}. Loading...")
        checkpoint = torch.load(checkpoint_file, map_location='cpu')
        
        accelerator.unwrap_model(model).load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint.get('scheduler_state_dict', scheduler.state_dict()))
        
        start_epoch = checkpoint.get('epoch', 0)
        start_step = checkpoint.get('step', 0) + 1
        
        # Check if the epoch was fully completed
        if start_step >= len(train_loader):
            start_epoch += 1
            start_step = 0
            accelerator.print(f"Epoch {start_epoch - 1} was complete. Advancing to Epoch {start_epoch}, Step 0")
            
        accelerator.print(f"Successfully resumed locally. Targeting Epoch {start_epoch}, starting at Step {start_step}")
        
    else:
        # 2. Try Hugging Face checkpoint if no local checkpoint is found
        accelerator.print("No local checkpoint found. Checking Hugging Face...")
        try:
            hf_checkpoint_path = hf_hub_download(
                repo_id="RinKana/AmadeusZero-114M", 
                filename="kaggle/working/amadeus_final.pt"
            )
            accelerator.print(f"Found HF checkpoint at {hf_checkpoint_path}. Loading...")
            checkpoint = torch.load(hf_checkpoint_path, map_location='cpu')
            
            accelerator.unwrap_model(model).load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            
            for param_group in optimizer.param_groups:
                param_group['lr'] = learning_rate
                param_group['initial_lr'] = learning_rate
            
            start_epoch = checkpoint.get('epoch', 0)
            start_step = checkpoint.get('step', 0) + 1
            
            # Check if the epoch was fully completed
            if start_step >= len(train_loader):
                start_epoch += 1
                start_step = 0
                accelerator.print(f"Epoch {start_epoch - 1} was complete. Advancing to Epoch {start_epoch}, Step 0")
                
            accelerator.print(f"Successfully loaded HF checkpoint. Resuming from Epoch {start_epoch}, Step {start_step}")
        except Exception as e:
            accelerator.print(f"No checkpoint found or download failed. Starting training from scratch.")

    @torch.no_grad()
    def estimate_loss(model, dataloader, eval_iters=50):
        model.eval()
        losses = torch.zeros(eval_iters, device=device)
        for k, (x, y) in enumerate(dataloader):
            if k >= eval_iters:
                break
            with accelerator.autocast():
                logits, loss = model(x, targets=y)
            # Gather loss from all devices
            loss_gathered = accelerator.gather(loss).mean()
            losses[k] = loss_gathered.item()
        model.train()
        return losses.mean().item()

    model.train()
    accelerator.print("Starting training loop...")
    
    for epoch in range(start_epoch, epochs):
        # Handle skipping steps if resuming
        if epoch == start_epoch and start_step > 0:
            active_dataloader = accelerator.skip_first_batches(train_loader, start_step)
        else:
            active_dataloader = train_loader

        # Only show progress bar on main process
        if accelerator.is_main_process:
            pbar = tqdm(active_dataloader, desc=f"Epoch {epoch}", total=len(train_loader), initial=start_step)
        else:
            pbar = active_dataloader
            
        for local_step, (x, y) in enumerate(pbar):
            # The actual step count for tracking
            step = local_step + start_step if epoch == start_epoch else local_step
            
            with accelerator.autocast():
                logits, loss = model(x, targets=y)
            
            # Using accelerate backward
            optimizer.zero_grad(set_to_none=True)
            accelerator.backward(loss)
            accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            # Calculate distributed metrics
            loss_gathered = accelerator.gather(loss).mean().item()
            train_ppl_val = math.exp(loss_gathered)
            
            if accelerator.is_main_process:
                pbar.set_postfix({"Loss": f"{loss_gathered:.4f}", "PPL": f"{train_ppl_val:.2f}"})
            
            if step % 1000 == 0 and step > 0:
                torch.cuda.empty_cache()
                val_loss = estimate_loss(model, val_loader, eval_iters=50)
                val_perplexity = math.exp(val_loss)
                current_lr = scheduler.get_last_lr()[0]
                
                # We can print this above the progress bar without breaking it
                if accelerator.is_main_process:
                    pbar.write(f"Validation at Step {step} -> Val Loss: {val_loss:.4f} | Val PPL: {val_perplexity:.4f} | LR: {current_lr:.6f}")
                
            is_last_step = step == len(train_loader) - 1
            
            # Ensure all GPUs reach this point before main process saves
            accelerator.wait_for_everyone()
            
            # Fix Deadlock: evaluate on all processes before saving
            if (step > 0 and step % 10000 == 0) or is_last_step:
                if step % 1000 != 0:
                    torch.cuda.empty_cache()
                    val_loss = estimate_loss(model, val_loader, eval_iters=50)
                    val_perplexity = math.exp(val_loss)

                if accelerator.is_main_process:
                    accelerator.print(f"Saving checkpoint at Epoch {epoch}, Step {step}...")
                    unwrapped_model = accelerator.unwrap_model(model)
                    
                    torch.save({
                        'epoch': epoch,
                        'step': step,
                        'model_state_dict': unwrapped_model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'loss': loss_gathered,
                        'perplexity': train_ppl_val,
                        'val_loss': val_loss,
                        'val_perplexity': val_perplexity,
                    }, checkpoint_file)
        
        # Reset start_step after the first resumed epoch
        start_step = 0

    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        accelerator.print("Training complete. Final checkpoint saved.")
        
    # Gracefully shut down the distributed process group to prevent NCCL warnings/hangs
    accelerator.end_training()

if __name__ == "__main__":
    main()


In [ ]:
!accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no train.py


In [ ]:
# --- INFERENCE ---
# We can import our model definition directly from the train.py script we created!
import torch
import torch.nn.functional as F
import tiktoken
from train import AmadeusZeroLM, Config

def generate_text(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_k=40, device="cuda"):
    model.eval()
    
    # Encode prompt
    tokens = tokenizer.encode(prompt, allowed_special="all")
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # If sequence gets too long, crop it to block_size
            idx_cond = x if x.size(1) <= model.config.block_size else x[:, -model.config.block_size:]
            
            # Forward pass
            logits, _ = model(idx_cond)
            logits = logits[:, -1, :] # Pluck the logits at the final step
            logits = logits / temperature
            
            # Top-K filtering
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
                
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            
            if idx_next.item() == 50256: # <|endoftext|> token
                break
            
            x = torch.cat((x, idx_next), dim=1)
            
    return tokenizer.decode(x[0].tolist())

# 1. Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = tiktoken.get_encoding("gpt2")
conf = Config()
model = AmadeusZeroLM(conf).to(device)

# 2. Load Checkpoint
try:
    print("Loading checkpoint amadeus_final.pt...")
    checkpoint = torch.load("amadeus_final.pt", map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded successfully! (Epoch {checkpoint.get('epoch', '?')}, Step {checkpoint.get('step', '?')})")
except Exception as e:
    print("Could not load checkpoint. The model will generate random garbage until trained.")
    print(f"Error: {e}")

# 3. Generate!
prompt = "Once upon a time, in a magical forest, "
print("\n--- Generating ---")
print(f"Prompt: {prompt}\n")
output = generate_text(model, tokenizer, prompt, max_new_tokens=150)
print(output)
